# Quantum Teleportation Simulation

**Day 25 | 30-Day Quantum Computing Challenge**

## Aim:
To simulate the three-qubit teleportation protocol and verify recovery of the input state after classical corrections.

## Tools:
Python, NumPy, Matplotlib (optional Qiskit)

## Expected Result:
Bob recovers the original input state with fidelity 1 in an ideal simulation.

**Execution:** Run the cells from top to bottom. The notebook uses an ideal simulator unless explicitly stated otherwise.

## 1. Protocol and state preparation

We choose a nontrivial input state

$$|\psi\rangle=\cos(\theta/2)|0\rangle+e^{i\phi}\sin(\theta/2)|1\rangle,\quad \theta=\pi/3,\;\phi=\pi/5.$$

The three qubits are ordered **q0 = Alice's input**, **q1 = Alice's Bell-pair half**, **q2 = Bob's half**. The simulation uses an 8-component complex statevector, not pre-sampled output bits.

In [ ]:
# Experiment step: execute this cell and inspect its output.
import numpy as np
import matplotlib.pyplot as plt

I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
zero = np.array([1, 0], dtype=complex)

def tensor3(a, b, c):
    return np.kron(np.kron(a, b), c)

def cnot(control, target):
    matrix = np.zeros((8, 8), dtype=complex)
    for col in range(8):
        bits = [(col >> 2) & 1, (col >> 1) & 1, col & 1]
        if bits[control]:
            bits[target] ^= 1
        row = (bits[0] << 2) | (bits[1] << 1) | bits[2]
        matrix[row, col] = 1
    return matrix

theta, phi = np.pi / 3, np.pi / 5
psi = np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])
print('Input amplitudes:', np.round(psi, 4))
print('Normalization:', np.vdot(psi, psi).real)

## 2. Entangle q1 and q2; perform Alice's Bell measurement

1. `H(q1)` and `CNOT(q1 → q2)` create the shared Bell pair.
2. `CNOT(q0 → q1)` and `H(q0)` prepare Alice's two-qubit measurement.
3. Measuring q0 and q1 gives one of four classical bit pairs, each with probability 1/4 for an ideal circuit.

In [ ]:
# Experiment step: execute this cell and inspect its output.
state = tensor3(psi, zero, zero)
for operation in [tensor3(I, H, I), cnot(1, 2), cnot(0, 1), tensor3(H, I, I)]:
    state = operation @ state

branch_data = {}
for a in (0, 1):
    for b in (0, 1):
        bob_unnormalized = state.reshape(2, 2, 2)[a, b, :]
        probability = float(np.vdot(bob_unnormalized, bob_unnormalized).real)
        bob_before = bob_unnormalized / np.sqrt(probability)
        # a is q0's measurement; b is q1's measurement.
        bob_after = (np.linalg.matrix_power(Z, a) @
                     np.linalg.matrix_power(X, b) @ bob_before)
        fidelity = float(abs(np.vdot(psi, bob_after))**2)
        branch_data[f'{a}{b}'] = {'probability': probability, 'fidelity': fidelity,
                                  'bob_state': bob_after}
        print(f'Bits {a}{b}: probability={probability:.3f}, fidelity={fidelity:.12f}')
assert all(np.isclose(item['fidelity'], 1.0) for item in branch_data.values())

## 3. Why Bob's corrections work

| Alice measures (q0,q1) | Bob applies |
|---|---|
| 00 | Identity |
| 01 | X |
| 10 | Z |
| 11 | X then Z (equivalently Z·X on the state) |

The fidelity $F=|\langle\psi|\psi_{Bob}\rangle|^2$ equals **1** for each ideal corrected branch. Alice's original state is consumed by the measurement; it is not copied.

## 4. Repeat the experiment and visualize Alice's outcomes

We sample the **computed** Born probabilities using a fixed seed for reproducibility. This is an ideal statevector-based simulation, not execution on physical hardware.

In [ ]:
# Experiment step: execute this cell and inspect its output.
shots = 1000
rng = np.random.default_rng(25)
labels = list(branch_data)
probabilities = [branch_data[k]['probability'] for k in labels]
samples = rng.choice(labels, size=shots, p=probabilities)
counts = {key: int(np.sum(samples == key)) for key in labels}
print('Measurement counts:', counts)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(list(counts), list(counts.values()), color=['#5369ac','#6d82c5','#9aa8dc','#bac5e7'])
ax.axhline(shots/4, linestyle='--', color='#dc8a39', label='Expected 25%')
ax.set(xlabel='Alice measurement bits (q0 q1)', ylabel='Counts',
       title='Quantum Teleportation: Alice Measurement Distribution')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('results.png', dpi=170)
plt.show()

## 5. Optional Qiskit circuit (requires `qiskit`)

This code builds a **dynamic circuit** with measurement-controlled corrections. Qiskit execution support for dynamic circuits depends on the backend. The core NumPy simulation above is complete independently.

In [ ]:
# Experiment step: execute this cell and inspect its output.
try:
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(3, 2)
    qc.initialize(psi, 0)
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)
    qc.measure(1, 1)
    with qc.if_test((qc.clbits[1], 1)):
        qc.x(2)
    with qc.if_test((qc.clbits[0], 1)):
        qc.z(2)
    print(qc.draw(output='text'))
except ImportError:
    print('Optional Qiskit circuit skipped. Install with: pip install qiskit')

## 6. Observations and limitations

- All four Alice outcomes are approximately equally likely; **each corrected branch** recovers the same input qubit state.
- Classical communication of two bits is required before Bob can apply the right correction.
- This is an **ideal simulator**: hardware noise, gate errors and classical latency are not modeled.
- The experiment demonstrates state transfer, **not faster-than-light communication**, cloning, or physical teleportation.

**Try it:** change `theta` and `phi`, rerun the notebook, and verify that all four fidelities remain approximately 1.

## Observation:
Record the measured or simulated outputs and compare them with the theoretical expectation. Note that finite-shot results can fluctuate.

## Conclusion:
The experiment demonstrates the quantum circuit or state transformation described above using simulation; it does not establish performance on physical quantum hardware.